# Marunthagam — Eval Results Analysis

Offline Tamil community health AI (Gemma 4 Good Hackathon)  
Analyzes triage performance, multi-seed variance, LoRA rank ablations, specialist comparisons, and Tamil fluency.

**Targets:**
- Triage F1 (overall) > 0.80  
- RED recall > 0.90  
- chrF++ > 0.60  

Run eval scripts first (`eval/scripts/run_eval.py`, `ablation_rank.py`) to populate `eval/results/` with real data.  
If no results are found, each section falls back to realistic synthetic demo data.

---
## Section 1 — Setup

Imports, constants, and the shared `load_latest_results()` helper.

In [ ]:
import json
import warnings
from IPython.display import display
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Path resolution ──────────────────────────────────────────────────────────
# Works whether run as a notebook or as a script.
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()  # Jupyter: cwd is usually the notebook dir

RESULTS_DIR = NOTEBOOK_DIR.parent / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Target thresholds ────────────────────────────────────────────────────────
THRESHOLD_F1        = 0.80   # overall weighted F1
THRESHOLD_RED_RECALL = 0.90  # RED class recall (safety-critical)
THRESHOLD_CHRF      = 0.60   # Tamil chrF++
HIGH_VARIANCE_STD   = 0.03   # flag if seed std exceeds this

# ── Colour palette ───────────────────────────────────────────────────────────
COLORS = {
    "GREEN":    "#2ecc71",
    "YELLOW":   "#f39c12",
    "RED":      "#e74c3c",
    "target":   "#2c3e50",
    "baseline": "#95a5a6",
    "fusion":   "#8e44ad",
    "triage":   "#3498db",
    "derm":     "#1abc9c",
    "maternal": "#e67e22",
}

SEEDS = [42, 137, 256]
LORA_RANKS = [4, 8, 16, 32, 64]
SPECIALISTS = ["triage", "derm", "maternal"]

# ── Global plot style ────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight"})


# ── Helper: load most-recent JSON matching a glob ────────────────────────────
def load_latest_results(pattern: str) -> dict | None:
    """
    Returns the parsed JSON from the most recently modified file in
    RESULTS_DIR whose name matches *pattern*, or None if no file exists.

    Example:
        data = load_latest_results("triage_eval_*.json")
    """
    candidates = sorted(
        RESULTS_DIR.glob(pattern),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not candidates:
        return None
    with candidates[0].open("r", encoding="utf-8") as fh:
        return json.load(fh)


print(f"Results dir : {RESULTS_DIR}")
print(f"Figures dir : {FIGURES_DIR}")
print(f"Results found: {len(list(RESULTS_DIR.glob('*.json')))} JSON file(s)")

### Tamil Font Note (Windows)

matplotlib does not render Tamil Unicode by default on Windows. Chart axes in this notebook use English labels to avoid rendering failures.

To enable Tamil labels in charts, install **Noto Sans Tamil** and configure matplotlib:

```python
import matplotlib.font_manager as fm
# Add the font path (adjust to your installation)
fm.fontManager.addfont('C:/Windows/Fonts/NotoSansTamil-Regular.ttf')
plt.rcParams['font.family'] = 'Noto Sans Tamil'
```

Download Noto Sans Tamil from https://fonts.google.com/noto/specimen/Noto+Sans+Tamil

---
## Section 2 — Triage Performance Overview

Per-class Precision / Recall / F1 for GREEN, YELLOW, RED.  
The bar chart shows F1 per class with a target line at **0.80**.  
RED recall is highlighted as PASS / FAIL vs the **0.90** safety target.

In [ ]:
# ── Load or synthesise triage eval data ──────────────────────────────────────
triage_data = load_latest_results("triage_eval_*.json")

if triage_data is None:
    print("⚠  No results found — showing DEMO data.")
    triage_data = {
        "per_class": {
            "GREEN":  {"precision": 0.86, "recall": 0.88, "f1": 0.87, "support": 120},
            "YELLOW": {"precision": 0.78, "recall": 0.75, "f1": 0.76, "support": 95},
            "RED":    {"precision": 0.87, "recall": 0.92, "f1": 0.88, "support": 65},
        },
        "weighted_f1": 0.832,
        "red_recall":  0.92,
        "mean_f1":     0.832,
        "std_f1":      0.018,
        "seeds":       SEEDS,
    }
else:
    print("Loaded real triage eval results.")

# ── Build styled DataFrame ────────────────────────────────────────────────────
per_class = triage_data["per_class"]
rows = []
for cls in ["GREEN", "YELLOW", "RED"]:
    m = per_class[cls]
    rows.append({
        "Class":     cls,
        "Precision": round(m["precision"], 3),
        "Recall":    round(m["recall"],    3),
        "F1":        round(m["f1"],        3),
        "Support":   m["support"],
    })

df_per_class = pd.DataFrame(rows).set_index("Class")

# Weighted F1 summary row
summary = pd.DataFrame(
    [{"Precision": "—", "Recall": "—",
      "F1": round(triage_data["weighted_f1"], 3),
      "Support": df_per_class["Support"].sum()}],
    index=["weighted avg"],
)
df_display = pd.concat([df_per_class, summary])


def _style_class(val, col):
    """Background-colour per triage class row."""
    colors_map = {"GREEN": "#d5f5e3", "YELLOW": "#fdebd0", "RED": "#fadbd8"}
    return f"background-color: {colors_map.get(col, '#f8f9fa')}"


styled = (
    df_display.style
    .set_caption("Triage Per-Class Performance")
    .apply(lambda s: [
        f"background-color: {'#d5f5e3' if s.name == 'GREEN' else '#fdebd0' if s.name == 'YELLOW' else '#fadbd8' if s.name == 'RED' else '#f8f9fa'}"
        for _ in s
    ], axis=1)
    .format(precision=3, na_rep="—")
)
display(styled)

# ── RED recall PASS/FAIL ──────────────────────────────────────────────────────
red_recall = triage_data["red_recall"]
pass_fail   = "PASS ✓" if red_recall >= THRESHOLD_RED_RECALL else "FAIL ✗"
colour_msg  = "\033[92m" if red_recall >= THRESHOLD_RED_RECALL else "\033[91m"
print(f"\n{colour_msg}RED Recall: {red_recall:.3f}  →  {pass_fail}  (target ≥ {THRESHOLD_RED_RECALL})\033[0m")
print(f"Weighted F1: {triage_data['weighted_f1']:.3f}  →  {'PASS ✓' if triage_data['weighted_f1'] >= THRESHOLD_F1 else 'FAIL ✗'}  (target ≥ {THRESHOLD_F1})")

In [ ]:
# ── Bar chart: F1 per class ───────────────────────────────────────────────────
classes = ["GREEN", "YELLOW", "RED"]
f1_vals = [per_class[c]["f1"] for c in classes]
bar_colors = [COLORS[c] for c in classes]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(classes, f1_vals, color=bar_colors, width=0.5, zorder=3, edgecolor="white", linewidth=1.5)

# Target line
ax.axhline(THRESHOLD_F1, color=COLORS["target"], linestyle="--", linewidth=1.5,
           label=f"Target F1 = {THRESHOLD_F1}")

# Value labels on bars
for bar, val in zip(bars, f1_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.008,
            f"{val:.3f}", ha="center", va="bottom", fontsize=11, fontweight="bold")

# RED recall callout
ax.annotate(
    f"RED recall = {red_recall:.3f}\n{'PASS ✓' if red_recall >= THRESHOLD_RED_RECALL else 'FAIL ✗'} (≥ {THRESHOLD_RED_RECALL})",
    xy=(2, per_class["RED"]["f1"]),
    xytext=(1.55, 0.65),
    fontsize=9,
    color=COLORS["RED"],
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=COLORS["RED"], lw=1.2),
)

ax.set_ylim(0, 1.05)
ax.set_ylabel("F1 Score")
ax.set_title(f"Triage F1 per Class  (target ≥ {THRESHOLD_F1}, weighted F1 = {triage_data['weighted_f1']:.3f})",
             fontsize=12)
ax.legend()
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "triage_f1_per_class.png")
plt.show()
print(f"Saved → {FIGURES_DIR / 'triage_f1_per_class.png'}")

---
## Section 3 — Confusion Matrix

Recall-normalised heatmap (each row sums to 1.0).  
Off-diagonal values show where the model misclassifies — the most safety-critical error is GREEN↔RED confusion.

In [ ]:
# ── Load or synthesise confusion matrix ──────────────────────────────────────
cm_data = load_latest_results("triage_confusion_*.json")

if cm_data is None:
    print("⚠  No confusion matrix results found — showing DEMO data.")
    # Rows = actual, cols = predicted  (GREEN, YELLOW, RED)
    # Realistic: GREEN mostly correct, YELLOW has most confusion, RED high recall
    cm_raw = np.array([
        [106,  12,   2],   # actual GREEN
        [ 10,  71,  14],   # actual YELLOW
        [  1,   4,  60],   # actual RED
    ], dtype=float)
else:
    print("Loaded real confusion matrix.")
    cm_raw = np.array(cm_data["matrix"], dtype=float)

# Normalise by actual class (row-wise recall normalisation)
cm_norm = cm_raw / cm_raw.sum(axis=1, keepdims=True)

labels = ["GREEN", "YELLOW", "RED"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=0, vmax=1,
    xticklabels=labels,
    yticklabels=labels,
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Recall (row-normalised)"},
)
ax.set_title("Triage Confusion Matrix — GREEN / YELLOW / RED", fontsize=12)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.tick_params(axis="both", labelsize=11)

# Highlight the RED recall cell border
ax.add_patch(plt.Rectangle((2, 2), 1, 1, fill=False, edgecolor=COLORS["RED"],
                            linewidth=3, clip_on=False))

fig.tight_layout()
fig.savefig(FIGURES_DIR / "triage_confusion_matrix.png")
plt.show()
print(f"Saved → {FIGURES_DIR / 'triage_confusion_matrix.png'}")

# Annotate worst confusion pair
off_diag = cm_norm.copy()
np.fill_diagonal(off_diag, 0)
worst_actual, worst_pred = np.unravel_index(off_diag.argmax(), off_diag.shape)
print(f"\nLargest off-diagonal: {labels[worst_actual]} predicted as {labels[worst_pred]}"
      f" — {off_diag[worst_actual, worst_pred]:.1%} of actual {labels[worst_actual]} cases")

---
## Section 4 — Multi-Seed Analysis

Training is run with three seeds (42, 137, 256) to verify result stability.  
High variance (std > 0.03) suggests the model is sensitive to initialisation — a reason to increase training steps or data.

Violin + scatter plot gives full distribution; table shows mean ± std per metric.

In [ ]:
# ── Load per-seed results (one JSON per seed, or a combined file) ─────────────
seed_results: list[dict] = []

for seed in SEEDS:
    data = load_latest_results(f"triage_eval_seed{seed}_*.json")
    if data:
        data["seed"] = seed
        seed_results.append(data)

# Also check a combined multi-seed file
if not seed_results:
    combined = load_latest_results("triage_multiseed_*.json")
    if combined:
        seed_results = combined.get("per_seed", [])

if not seed_results:
    print("⚠  No per-seed results found — showing DEMO data.")
    seed_results = [
        {"seed": 42,  "weighted_f1": 0.832, "red_recall": 0.923, "chrf": 0.638},
        {"seed": 137, "weighted_f1": 0.818, "red_recall": 0.908, "chrf": 0.621},
        {"seed": 256, "weighted_f1": 0.845, "red_recall": 0.938, "chrf": 0.649},
    ]
else:
    print(f"Loaded {len(seed_results)} seed result(s).")

df_seeds = pd.DataFrame(seed_results)

# ── Summary table ─────────────────────────────────────────────────────────────
metrics = [c for c in ["weighted_f1", "red_recall", "chrf"] if c in df_seeds.columns]

summary_rows = []
for metric in metrics:
    vals = df_seeds[metric].astype(float)
    mean, std = vals.mean(), vals.std(ddof=0)
    flag = " ← HIGH VARIANCE" if std > HIGH_VARIANCE_STD else ""
    summary_rows.append({
        "Metric":    metric,
        "Mean":      round(mean, 4),
        "Std":       round(std, 4),
        "Min":       round(vals.min(), 4),
        "Max":       round(vals.max(), 4),
        "Note":      flag.strip(),
    })

df_summary = pd.DataFrame(summary_rows).set_index("Metric")

def _highlight_variance(row):
    return ["background-color: #fde8e8" if row["Note"] else "" for _ in row]

display(
    df_summary.style
    .set_caption(f"Multi-Seed Summary (seeds: {SEEDS}, high-variance threshold: std > {HIGH_VARIANCE_STD})")
    .apply(_highlight_variance, axis=1)
    .format(precision=4)
)

In [ ]:
# ── Violin + scatter: F1 distribution across seeds ────────────────────────────
if len(metrics) == 0:
    print("No numeric metrics to plot.")
else:
    df_long = df_seeds.melt(id_vars="seed", value_vars=metrics,
                            var_name="Metric", value_name="Score")

    fig, ax = plt.subplots(figsize=(8, 4))

    # Violin with inner box
    sns.violinplot(
        data=df_long, x="Metric", y="Score",
        inner="box", palette=["#3498db", COLORS["RED"], COLORS["YELLOW"]],
        cut=0, ax=ax, alpha=0.7,
    )

    # Scatter individual seed points
    sns.stripplot(
        data=df_long, x="Metric", y="Score",
        color="black", size=8, jitter=True, ax=ax, zorder=5,
    )

    # Threshold reference lines
    thresholds = {
        "weighted_f1": THRESHOLD_F1,
        "red_recall":  THRESHOLD_RED_RECALL,
        "chrf":        THRESHOLD_CHRF,
    }
    x_positions = {m: i for i, m in enumerate(df_long["Metric"].unique())}
    for metric, thr in thresholds.items():
        if metric in x_positions:
            xi = x_positions[metric]
            ax.plot([xi - 0.35, xi + 0.35], [thr, thr],
                    color=COLORS["target"], linestyle="--", linewidth=1.5, zorder=4)
            ax.text(xi + 0.37, thr, f"target={thr}",
                    va="center", fontsize=8, color=COLORS["target"])

    ax.set_ylim(0.4, 1.05)
    ax.set_title(f"Score Distribution Across {len(SEEDS)} Seeds — Dots = individual seed values",
                 fontsize=11)
    ax.set_ylabel("Score")
    ax.set_xlabel("")

    # Annotate HIGH VARIANCE
    for row in summary_rows:
        if row["Note"] and row["Metric"] in x_positions:
            xi = x_positions[row["Metric"]]
            ax.text(xi, 0.42, "HIGH VARIANCE", ha="center", fontsize=8,
                    color="#c0392b", fontweight="bold")

    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "multiseed_violin.png")
    plt.show()
    print(f"Saved → {FIGURES_DIR / 'multiseed_violin.png'}")

---
## Section 5 — LoRA Rank Ablation

Ranks tested: **4, 8, 16, 32, 64**.  
The line plot shows how F1 and RED recall change with rank (log x-axis, with error bars across seeds).  
The minimum rank that clears the F1 > 0.80 target is annotated — this informs the sweet spot between model size and performance.

In [ ]:
# ── Load or synthesise rank ablation data ─────────────────────────────────────
rank_data = load_latest_results("ablation_rank_comparison.json")

if rank_data is None:
    print("⚠  No rank ablation results found — showing DEMO data.")
    # Realistic: lower ranks underfit (F1 ~0.72), higher ranks plateau (~0.84)
    rank_data = {
        "ranks": LORA_RANKS,
        "mean_f1":         [0.712, 0.758, 0.810, 0.832, 0.838],
        "std_f1":          [0.024, 0.021, 0.018, 0.016, 0.015],
        "mean_red_recall": [0.813, 0.861, 0.898, 0.920, 0.931],
        "std_red_recall":  [0.031, 0.027, 0.022, 0.019, 0.016],
    }
else:
    print("Loaded real rank ablation results.")

ranks          = np.array(rank_data["ranks"])
mean_f1        = np.array(rank_data["mean_f1"])
std_f1         = np.array(rank_data["std_f1"])
mean_red_recall = np.array(rank_data["mean_red_recall"])
std_red_recall  = np.array(rank_data["std_red_recall"])

# ── Find minimum rank for F1 > threshold ──────────────────────────────────────
passing_indices = np.where(mean_f1 >= THRESHOLD_F1)[0]
min_passing_rank = ranks[passing_indices[0]] if len(passing_indices) > 0 else None

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

# — F1 vs rank —
ax1.semilogx(ranks, mean_f1, marker="o", color=COLORS["triage"],
             linewidth=2, markersize=8, label="Mean F1 (3 seeds)", base=2)
ax1.fill_between(ranks, mean_f1 - std_f1, mean_f1 + std_f1,
                 alpha=0.2, color=COLORS["triage"])
ax1.axhline(THRESHOLD_F1, color=COLORS["target"], linestyle="--", linewidth=1.5,
            label=f"Target = {THRESHOLD_F1}")

if min_passing_rank is not None:
    min_idx = list(ranks).index(min_passing_rank)
    ax1.annotate(
        f"Min rank for F1 > {THRESHOLD_F1}\n→ rank {min_passing_rank}",
        xy=(min_passing_rank, mean_f1[min_idx]),
        xytext=(min_passing_rank * 1.5, mean_f1[min_idx] - 0.04),
        fontsize=9, color=COLORS["triage"],
        arrowprops=dict(arrowstyle="->", color=COLORS["triage"], lw=1.2),
    )

ax1.set_xscale("log", base=2)
ax1.set_xticks(ranks)
ax1.set_xticklabels(ranks)
ax1.set_xlabel("LoRA Rank (log₂ scale)")
ax1.set_ylabel("Weighted F1")
ax1.set_title(f"F1 vs LoRA Rank  (target ≥ {THRESHOLD_F1})")
ax1.legend()
ax1.set_ylim(0.65, 0.95)

# — RED recall vs rank —
ax2.semilogx(ranks, mean_red_recall, marker="s", color=COLORS["RED"],
             linewidth=2, markersize=8, label="Mean RED Recall (3 seeds)", base=2)
ax2.fill_between(ranks, mean_red_recall - std_red_recall, mean_red_recall + std_red_recall,
                 alpha=0.2, color=COLORS["RED"])
ax2.axhline(THRESHOLD_RED_RECALL, color=COLORS["target"], linestyle="--", linewidth=1.5,
            label=f"Target = {THRESHOLD_RED_RECALL}")

ax2.set_xscale("log", base=2)
ax2.set_xticks(ranks)
ax2.set_xticklabels(ranks)
ax2.set_xlabel("LoRA Rank (log₂ scale)")
ax2.set_ylabel("RED Recall")
ax2.set_title(f"RED Recall vs LoRA Rank  (target ≥ {THRESHOLD_RED_RECALL})")
ax2.legend()
ax2.set_ylim(0.75, 1.0)

fig.suptitle("LoRA Rank Ablation — Triage Specialist", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "lora_rank_ablation.png")
plt.show()
print(f"Saved → {FIGURES_DIR / 'lora_rank_ablation.png'}")
if min_passing_rank:
    print(f"Minimum rank achieving F1 > {THRESHOLD_F1}: rank {min_passing_rank}")
else:
    print(f"No rank achieves F1 > {THRESHOLD_F1} — increase max rank or training steps.")

---
## Section 6 — Specialist Comparison

Compares the three domain specialists (triage, derm, maternal) against the generalist baseline and the KALAVAI fusion model.

A dotted baseline line shows the generalist's weighted F1.  
The fusion bar (purple) is the expected winner — if it doesn't beat the best specialist by +3%, that should be reported honestly as a finding.

In [ ]:
# ── Load or synthesise specialist comparison data ─────────────────────────────
specialist_data = load_latest_results("specialist_comparison_*.json")

if specialist_data is None:
    print("⚠  No specialist comparison results found — showing DEMO data.")
    specialist_data = {
        "generalist_f1": 0.762,
        "specialists": {
            "triage":   {"mean_f1": 0.832, "std_f1": 0.018},
            "derm":     {"mean_f1": 0.801, "std_f1": 0.022},
            "maternal": {"mean_f1": 0.815, "std_f1": 0.019},
        },
        "fusion_f1":     0.861,
        "fusion_available": True,
    }
else:
    print("Loaded real specialist comparison results.")

generalist_f1 = specialist_data["generalist_f1"]
specs         = specialist_data["specialists"]
fusion_f1     = specialist_data.get("fusion_f1")
fusion_avail  = specialist_data.get("fusion_available", fusion_f1 is not None)

# Build bar data
names  = list(SPECIALISTS)
means  = [specs[s]["mean_f1"] for s in SPECIALISTS]
stds   = [specs[s]["std_f1"]  for s in SPECIALISTS]
colors = [COLORS[s] for s in SPECIALISTS]

if fusion_avail and fusion_f1 is not None:
    names.append("KALAVAI fusion")
    means.append(fusion_f1)
    stds.append(0.0)
    colors.append(COLORS["fusion"])

x = np.arange(len(names))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(x, means, yerr=stds, color=colors, width=0.55, capsize=5,
              error_kw={"elinewidth": 1.5, "ecolor": "#555"}, zorder=3,
              edgecolor="white", linewidth=1.5)

# Generalist dotted baseline
ax.axhline(generalist_f1, color=COLORS["baseline"], linestyle=":", linewidth=2,
           label=f"Generalist baseline = {generalist_f1:.3f}")

# F1 > 0.80 target
ax.axhline(THRESHOLD_F1, color=COLORS["target"], linestyle="--", linewidth=1.5,
           label=f"Target F1 = {THRESHOLD_F1}")

# Value labels
for bar, mean, std in zip(bars, means, stds):
    label = f"{mean:.3f}"
    if std > 0:
        label += f"\n±{std:.3f}"
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (std or 0) + 0.008,
            label, ha="center", va="bottom", fontsize=9, fontweight="bold")

# Fusion gain annotation
if fusion_avail and fusion_f1 is not None:
    best_specialist_f1 = max(specs[s]["mean_f1"] for s in SPECIALISTS)
    gain = fusion_f1 - best_specialist_f1
    gain_label = f"+{gain:.1%} vs best specialist"
    ax.annotate(gain_label,
                xy=(len(names) - 1, fusion_f1),
                xytext=(len(names) - 1.5, fusion_f1 + 0.03),
                fontsize=9, color=COLORS["fusion"], fontweight="bold",
                arrowprops=dict(arrowstyle="->", color=COLORS["fusion"], lw=1.2))

ax.set_xticks(x)
ax.set_xticklabels([n.capitalize() for n in names], fontsize=11)
ax.set_ylim(0.65, 1.0)
ax.set_ylabel("Weighted F1 (mean ± std, 3 seeds)")
ax.set_title("Specialist vs Generalist vs KALAVAI Fusion  (target F1 ≥ 0.80)", fontsize=12)
ax.legend()
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "specialist_comparison.png")
plt.show()
print(f"Saved → {FIGURES_DIR / 'specialist_comparison.png'}")

# Specialist gain check
print("\nPer-specialist gain over generalist:")
for spec in SPECIALISTS:
    gain = specs[spec]["mean_f1"] - generalist_f1
    status = "PASS ✓" if gain >= 0.05 else "below +5% target"
    print(f"  {spec:12s}: +{gain:.1%}  {status}")

---
## Section 7 — Tamil Fluency (chrF++)

chrF++ measures character-level n-gram overlap between model outputs and human-written Tamil reference responses.  
A score above **0.60** indicates fluent, clinically appropriate Tamil.  
Note: chrF++ does not measure medical accuracy — use triage F1 for that.

In [ ]:
# ── Load or synthesise chrF++ data ────────────────────────────────────────────
chrf_data = load_latest_results("chrf_eval_*.json")

if chrf_data is None:
    print("⚠  No chrF++ results found — showing DEMO data.")
    chrf_data = {
        "specialists": {
            "triage":   {"mean_chrf": 0.638, "std_chrf": 0.021},
            "derm":     {"mean_chrf": 0.612, "std_chrf": 0.025},
            "maternal": {"mean_chrf": 0.651, "std_chrf": 0.019},
        },
        "generalist_chrf": 0.574,
    }
else:
    print("Loaded real chrF++ results.")

chrf_specs     = chrf_data["specialists"]
generalist_chrf = chrf_data.get("generalist_chrf")

spec_names  = list(SPECIALISTS)
chrf_means  = [chrf_specs[s]["mean_chrf"] for s in SPECIALISTS]
chrf_stds   = [chrf_specs[s]["std_chrf"]  for s in SPECIALISTS]
chrf_colors = [COLORS[s] for s in SPECIALISTS]

x = np.arange(len(spec_names))

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(x, chrf_means, yerr=chrf_stds, color=chrf_colors,
              width=0.5, capsize=5, error_kw={"elinewidth": 1.5, "ecolor": "#555"},
              zorder=3, edgecolor="white", linewidth=1.5)

# Target line
ax.axhline(THRESHOLD_CHRF, color=COLORS["target"], linestyle="--", linewidth=1.5,
           label=f"Target chrF++ = {THRESHOLD_CHRF}")

# Generalist baseline (if available)
if generalist_chrf is not None:
    ax.axhline(generalist_chrf, color=COLORS["baseline"], linestyle=":", linewidth=2,
               label=f"Generalist = {generalist_chrf:.3f}")

# Value labels
for bar, mean, std in zip(bars, chrf_means, chrf_stds):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + std + 0.006,
            f"{mean:.3f}\n±{std:.3f}", ha="center", va="bottom",
            fontsize=9, fontweight="bold")

# PASS/FAIL badges
for xi, (mean, std) in enumerate(zip(chrf_means, chrf_stds)):
    badge = "PASS ✓" if mean >= THRESHOLD_CHRF else "FAIL ✗"
    badge_color = "#27ae60" if mean >= THRESHOLD_CHRF else "#c0392b"
    ax.text(xi, 0.41, badge, ha="center", fontsize=9,
            color=badge_color, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in spec_names], fontsize=11)
ax.set_ylim(0.38, 0.78)
ax.set_ylabel("chrF++ Score")
ax.set_title(f"Tamil Fluency — chrF++ per Specialist  (target ≥ {THRESHOLD_CHRF})", fontsize=12)
ax.legend(loc="upper left")
ax.yaxis.grid(True, zorder=0)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig(FIGURES_DIR / "chrf_fluency.png")
plt.show()
print(f"Saved → {FIGURES_DIR / 'chrf_fluency.png'}")
print("\nNote: Generated text evaluated against human-written Tamil reference responses.")
print("chrF++ measures character n-gram overlap — it is a proxy for fluency, not medical accuracy.")

---
## Summary

All figures saved to `eval/results/figures/`.  
Replace synthetic demo data by running the eval scripts and placing JSON output in `eval/results/`.

| Expected file | Produces |
|---|---|
| `triage_eval_*.json` | Sections 2, 3 (per-class), Section 4 |
| `triage_confusion_*.json` | Section 3 (confusion matrix) |
| `triage_eval_seed{42,137,256}_*.json` | Section 4 (per-seed) |
| `ablation_rank_comparison.json` | Section 5 |
| `specialist_comparison_*.json` | Section 6 |
| `chrf_eval_*.json` | Section 7 |

In [ ]:
# ── Final: list all saved figures ─────────────────────────────────────────────
saved = sorted(FIGURES_DIR.glob("*.png"))
if saved:
    print(f"Figures written to {FIGURES_DIR}:")
    for f in saved:
        print(f"  {f.name}")
else:
    print("No figures found — run all cells above first.")